# Agrixore — Dataset Builder, MobileNetV3 Trainer & Deployment Export

This notebook builds a 13-class wheat/corn/cotton leaf image dataset, validates image files and labels, removes exact duplicates, groups near-duplicates to reduce split leakage, creates train/validation/test splits, trains MobileNetV3-Small, evaluates the held-out test set, and exports TorchScript + ONNX artifacts.

**Important — run order:** In Google Colab, use **Runtime → Restart session**, then **Runtime → Run all**. Do not jump directly to a later cell. The setup cell defines all shared imports and helper functions. If a cell stops with an error, fix it before continuing.

**Sources and licensing:** the cotton Kaggle dataset listing reports its license as Unknown; check source terms before redistribution. The wheat Zenodo record contains a 999-image subset. Source-reported counts may differ from the cleaned dataset.

**Execution environment:** designed for Google Colab. Dataset downloads, pretrained weights, and training require internet access; a GPU is recommended. This notebook has been statically reviewed, but has not been run end-to-end here. Successful notebook execution does not by itself prove field-level diagnostic accuracy.


In [ ]:
# Run this cell first. Colab normally includes compatible torch/torchvision builds.
%pip -q install datasets kagglehub pillow imagehash pandas scikit-learn tqdm matplotlib onnx onnxruntime


In [ ]:
from pathlib import Path
import os, re, json, hashlib, zipfile, subprocess, time, random, platform, shutil, copy
from PIL import Image, ImageFile
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
import imagehash

ImageFile.LOAD_TRUNCATED_IMAGES = False

ROOT = Path('/content/agrixore_work')
RAW = ROOT / 'raw'
CLEAN = ROOT / 'Agrixore_Dataset'
META = CLEAN / 'metadata'
for p in (RAW, CLEAN, META):
    p.mkdir(parents=True, exist_ok=True)

# Recover the partial wheat ZIP created by the earlier troubleshooting steps, if present.
legacy_wheat_zip = Path('/content/agrixore_raw/wheat_999.zip')
current_wheat_zip = RAW / 'wheat_999.zip'
if legacy_wheat_zip.exists() and not current_wheat_zip.exists():
    shutil.copy2(legacy_wheat_zip, current_wheat_zip)
    print('Recovered previous partial wheat download for resume.')

SEED = 42
EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
TARGET = [
    'corn_healthy', 'corn_blight', 'corn_common_rust', 'corn_gray_leaf_spot',
    'cotton_healthy', 'cotton_bacterial_blight', 'cotton_curl_virus', 'cotton_fusarium_wilt',
    'wheat_healthy', 'wheat_yellow_rust', 'wheat_brown_rust', 'wheat_septoria', 'wheat_mildew'
]

def imgs(root):
    return [p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in EXT]

def norm(x):
    return re.sub(r'[^a-z0-9]+', ' ', str(x).lower()).strip()

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

print('Setup complete.')
print('Workspace:', ROOT)
print('IMPORTANT: In Colab, use Runtime > Run all from the top in a fresh runtime.')


## 1 — Download sources

In [ ]:
# 1. Download sources
# IMPORTANT: run setup cell above first, or use Runtime > Run all.
if 'RAW' not in globals() or 'imgs' not in globals() or 'load_dataset' not in globals():
    raise RuntimeError("Setup is missing. In Colab choose Runtime > Run all from the top.")

# Corn — Hugging Face
corn = load_dataset('Project-AgML/corn_maize_leaf_disease', split='train')
corn_dir = RAW / 'corn_hf'
corn_dir.mkdir(parents=True, exist_ok=True)
for i, row in enumerate(tqdm(corn, desc='Exporting corn')):
    lab = row['label']
    if isinstance(lab, int):
        lab = corn.features['label'].names[lab]
    image = row['image']
    if image is None:
        continue
    folder = corn_dir / str(lab)
    folder.mkdir(parents=True, exist_ok=True)
    image.convert('RGB').save(folder / f'corn_{i:06d}.jpg', quality=95)
print('Corn images exported:', len(imgs(corn_dir)))

# Cotton — Kaggle
import kagglehub
cotton_dir = Path(kagglehub.dataset_download('seroshkarim/cotton-leaf-disease-dataset'))
print('Cotton dataset downloaded to:', cotton_dir)
print('Cotton image files found:', len(imgs(cotton_dir)))

# Wheat — Zenodo. The ZIP is about 1 GB; resume partial downloads where possible.
wzip = RAW / 'wheat_999.zip'
url = 'https://zenodo.org/records/7573133/files/Long%202023%20Plant%20Path%20999%20photos.zip?download=1'
expected_md5 = '1f8a7c660c2da137b957fb4edee3dd7d'

def file_md5(path):
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def download_wheat_zip(resume=True):
    # Only request resume when a partial file actually exists.
    cmd = ['curl', '-fL', '--retry', '5', '--retry-delay', '3',
           '--retry-all-errors']
    if resume and wzip.exists() and wzip.stat().st_size > 0:
        cmd += ['-C', '-']
    cmd += ['-o', str(wzip), url]
    result = subprocess.run(cmd)
    return result.returncode == 0

# Try to use an existing verified archive. If its checksum is wrong, remove it
# and make one clean download attempt so rerunning this cell cannot loop forever.
verified_zip = False
for attempt in range(2):
    if not (wzip.exists() and zipfile.is_zipfile(wzip)):
        if not download_wheat_zip(resume=True):
            if wzip.exists():
                wzip.unlink()
            if not download_wheat_zip(resume=False):
                raise RuntimeError('Wheat ZIP download failed after a clean retry. Check internet access and rerun this cell.')
    if not zipfile.is_zipfile(wzip):
        if wzip.exists():
            wzip.unlink()
        if attempt == 0:
            continue
        raise RuntimeError('Wheat download is not a valid ZIP after retry. Check the connection and rerun.')
    actual_md5 = file_md5(wzip)
    if actual_md5 == expected_md5:
        verified_zip = True
        break
    print(f'Wheat ZIP checksum mismatch on attempt {attempt + 1}: expected {expected_md5}, got {actual_md5}.')
    wzip.unlink(missing_ok=True)
    if attempt == 0:
        print('Downloading the wheat archive again from the beginning...')
        if not download_wheat_zip(resume=False):
            raise RuntimeError('Wheat ZIP download failed during checksum-recovery retry. Rerun this cell when internet is stable.')

if not verified_zip:
    raise RuntimeError('Wheat ZIP checksum still does not match after retry. Do not extract this archive; check the Zenodo file/checksum.')
print('Wheat ZIP checksum verified:', expected_md5)

wdir = RAW / 'wheat_extracted'
extraction_marker = wdir / '.extraction_complete'
if not extraction_marker.exists():
    # If a previous extraction was interrupted, remove it and extract cleanly.
    if wdir.exists():
        shutil.rmtree(wdir)
    wdir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(wzip) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt member in wheat ZIP: {bad_member}")
        archive.extractall(wdir)
    extraction_marker.write_text('complete')
print('Wheat images extracted:', len(imgs(wdir)))
if len(imgs(wdir)) == 0:
    raise RuntimeError("No wheat images found after extraction. Inspect the ZIP folder structure before continuing.")


## 2 — Build and verify the source manifest

Corn and cotton labels are read from their class folders; wheat labels are read from filename prefixes (for example, `YellowRust1118.png`). The code deliberately stops on unknown labels and never silently guesses a disease class.

In [ ]:
# 2. Build and verify the source manifest.
if any(name not in globals() for name in ('RAW', 'TARGET', 'imgs', 'norm', 'pd')):
    raise RuntimeError("Setup is missing. In Colab choose Runtime > Run all from the top.")

def target(crop, source_label):
    n = norm(source_label)
    aliases = {
        'corn': {
            'healthy': 'corn_healthy',
            'blight': 'corn_blight',
            'common rust': 'corn_common_rust',
            'gray leaf spot': 'corn_gray_leaf_spot',
            'grey leaf spot': 'corn_gray_leaf_spot',
        },
        'cotton': {
            'healthy': 'cotton_healthy',
            'curl virus': 'cotton_curl_virus',
            'bacterial blight': 'cotton_bacterial_blight',
            'fusarium wilt': 'cotton_fusarium_wilt',
            # The downloaded source folder contains this misspelling.
            'fussarium wilt': 'cotton_fusarium_wilt',
        },
        'wheat': {
            'healthy': 'wheat_healthy',
            'yellow rust': 'wheat_yellow_rust',
            'brown rust': 'wheat_brown_rust',
            'septoria': 'wheat_septoria',
            'mildew': 'wheat_mildew',
        }
    }
    for alias, cls in aliases[crop].items():
        if n == alias or n.endswith(' ' + alias) or n.startswith(alias + ' ') or (' ' + alias + ' ') in (' ' + n + ' '):
            return cls
    return None

def find_class_from_path(crop, path, root):
    path = Path(path)
    rel = path.relative_to(Path(root))

    # Zenodo wheat files are stored together in one directory. Their filenames
    # encode the class, e.g. YellowRust1118.png and Healthy2161.png.
    if crop == 'wheat':
        stem = path.stem
        patterns = [
            (r'^YellowRust(?:\d|[_\-\s.]|$)', 'YellowRust', 'wheat_yellow_rust'),
            (r'^BrownRust(?:\d|[_\-\s.]|$)', 'BrownRust', 'wheat_brown_rust'),
            (r'^Septoria(?:\d|[_\-\s.]|$)', 'Septoria', 'wheat_septoria'),
            (r'^Mildew(?:\d|[_\-\s.]|$)', 'Mildew', 'wheat_mildew'),
            (r'^Healthy(?:\d|[_\-\s.]|$)', 'Healthy', 'wheat_healthy'),
        ]
        for pattern, source_label, class_name in patterns:
            if re.match(pattern, stem, flags=re.IGNORECASE):
                return source_label, class_name
        # Do not guess if a wheat filename does not have a recognized prefix.
        return 'UNKNOWN_FILENAME:' + stem, None

    # Corn and cotton classes are encoded in folder names; inspect nearest parent first.
    candidates = list(rel.parts[:-1])[::-1]
    for part in candidates:
        cls = target(crop, part)
        if cls is not None:
            return part, cls
    return (candidates[0] if candidates else 'UNKNOWN'), None

records = []
for crop, source, root in [
    ('corn', 'Project-AgML/corn_maize_leaf_disease', corn_dir),
    ('cotton', 'Kaggle/seroshkarim/cotton-leaf-disease-dataset', cotton_dir),
    ('wheat', 'Zenodo/7573133', wdir)
]:
    for path in imgs(root):
        source_class, mapped_class = find_class_from_path(crop, path, root)
        records.append({
            'crop': crop, 'source': source, 'source_class': source_class,
            'class': mapped_class, 'original_path': str(path)
        })

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError("No image files found in the downloaded datasets.")
print('Candidate image files:', len(df))
print('Source label counts:')
display(df.groupby(['crop', 'source_class'], dropna=False).size().reset_index(name='count'))
print('Mapped target class counts:')
display(df.groupby(['crop', 'class'], dropna=False).size().reset_index(name='count'))

unknown = df[df['class'].isna()]
if len(unknown):
    print('STOP: these source labels/filenames did not map to a known disease class:')
    display(unknown.groupby(['crop', 'source_class']).size().reset_index(name='count'))
    raise ValueError(
        "Unknown source labels detected. Do not guess labels. Review the displayed folders/filenames and update the explicit mapping."
    )

found_classes = set(df['class'])
missing_classes = set(TARGET) - found_classes
if missing_classes:
    raise ValueError(f"Required classes missing from the downloaded data: {sorted(missing_classes)}")

# Sanity check: all 999 wheat files should map to the five filename-defined classes.
wheat_counts = df[df.crop == 'wheat']['class'].value_counts().to_dict()
print('Wheat filename-derived class counts:', wheat_counts)
if len(df[df.crop == 'wheat']) != 999:
    print('WARNING: expected 999 wheat images; actual count differs. Confirm the extracted source dataset.')
print('All 13 expected classes mapped successfully.')


## 3 — Validate, deduplicate, and audit images

In [ ]:
valid=[]; bad=[]
for _,r in tqdm(df.iterrows(),total=len(df),desc='Validating'):
    try:
        with Image.open(r.original_path) as im:
            im.verify()
        with Image.open(r.original_path) as im:
            rgb=im.convert('RGB'); w,h=rgb.size
        if w<32 or h<32: raise ValueError(f'too small: {w}x{h}')
        d=r.to_dict(); d.update(width=w,height=h,sha256=sha(r.original_path)); valid.append(d)
    except Exception as e:
        bad.append({**r.to_dict(),'error':repr(e)})
valid=pd.DataFrame(valid); bad=pd.DataFrame(bad)
bad.to_csv(META/'corrupted_images.csv',index=False)
if valid.empty:
    raise RuntimeError("No valid images remain after image validation. Review metadata/corrupted_images.csv.")

conf=valid.groupby('sha256')['class'].nunique()
conflict=conf[conf>1].index
conflict_rows=valid[valid.sha256.isin(conflict)].copy()
conflict_rows.to_csv(META/'conflicting_label_exact_duplicates.csv',index=False)

if len(conflict):
    print(f'WARNING: found {len(conflict)} exact image hashes assigned to multiple classes.')
    print(f'Quarantining {len(conflict_rows)} records rather than guessing which label is correct.')
    display(conflict_rows[['crop','source_class','class','original_path','sha256']])
    valid=valid[~valid.sha256.isin(conflict)].copy()
else:
    print('No exact-image label conflicts found.')

if valid.empty:
    raise RuntimeError('No images remain after quarantining exact-image label conflicts. Review conflicting_label_exact_duplicates.csv.')

dups=valid[valid.duplicated('sha256',False)].sort_values('sha256')
dups.to_csv(META/'exact_duplicate_groups.csv',index=False)
clean=valid.drop_duplicates('sha256').copy()
print('Valid before duplicate checks:',len(pd.DataFrame(valid))+len(conflict_rows),
      'Conflicting-label records quarantined:',len(conflict_rows),
      'Corrupt/invalid:',len(bad),'Same-label exact duplicates removed:',len(valid)-len(clean),
      'Clean images:',len(clean))


## 4 — Flag near-duplicates

Near-duplicates are reported, not blindly deleted. This avoids accidentally removing legitimate images. Review `near_duplicate_candidates.csv` if many candidates appear.

In [ ]:
ph=[]
for _,r in tqdm(clean.iterrows(),total=len(clean),desc='Perceptual hashes'):
    with Image.open(r.original_path) as im:
        ph.append({'sha256':r.sha256,'class':r['class'],'phash':str(imagehash.phash(im.convert('RGB')))})
ph=pd.DataFrame(ph)
near=[]
for cls,g in ph.groupby('class'):
    a=list(g.itertuples(index=False))
    hashes=[imagehash.hex_to_hash(x.phash) for x in a]
    for i in range(len(a)):
        for j in range(i+1,len(a)):
            d=hashes[i]-hashes[j]
            if d<=6:
                near.append({'class':cls,'sha256_a':a[i].sha256,'sha256_b':a[j].sha256,'hamming_distance':int(d)})
near=pd.DataFrame(near,columns=['class','sha256_a','sha256_b','hamming_distance'])
near.to_csv(META/'near_duplicate_candidates.csv',index=False)
print('Near-duplicate candidates flagged:',len(near))

## 5 — Group-aware 70/15/15 split

Images linked as near-duplicates are kept in the same split to reduce leakage. The code checks that every class has enough independent groups for non-empty training, validation, and test subsets. The actual proportions can vary slightly because groups, rather than individual images, are split.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Build connected components from perceptual-hash pairs so near-duplicate images
# are assigned to one split rather than leaking across train/validation/test.
parent={h:h for h in clean['sha256']}
def find(x):
    while parent[x] != x:
        parent[x]=parent[parent[x]]
        x=parent[x]
    return x
def union(a,b):
    ra,rb=find(a),find(b)
    if ra != rb: parent[rb]=ra
if len(near):
    for r in near.itertuples(index=False): union(r.sha256_a,r.sha256_b)
clean=clean.copy()
clean['group_id']=[find(h) for h in clean['sha256']]

parts=[]
for cls,g in clean.groupby('class',sort=True):
    if g['group_id'].nunique()<4:
        raise ValueError(f'Class {cls} has fewer than 4 independent image groups; cannot create non-empty train/validation/test splits.')
    splitter1=GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=SEED)
    tr_idx,tmp_idx=next(splitter1.split(g,groups=g['group_id']))
    tr=g.iloc[tr_idx].copy(); tmp=g.iloc[tmp_idx].copy()
    splitter2=GroupShuffleSplit(n_splits=1,test_size=0.50,random_state=SEED)
    va_idx,te_idx=next(splitter2.split(tmp,groups=tmp['group_id']))
    va=tmp.iloc[va_idx].copy(); te=tmp.iloc[te_idx].copy()
    for x,s in [(tr,'train'),(va,'val'),(te,'test')]:
        x['split']=s; parts.append(x)
final=pd.concat(parts,ignore_index=True)
# Assert no group can occur in more than one split.
assert final.groupby('group_id')['split'].nunique().max()==1
assert set(final['class'])==set(TARGET)
final.to_csv(META/'dataset_manifest.csv',index=False)
split_table = pd.crosstab(final['class'], final['split']).reindex(index=sorted(TARGET), columns=['train','val','test'], fill_value=0)
display(split_table)
print('Split sizes:',final['split'].value_counts().to_dict())
if (split_table == 0).any().any():
    zero_cells = [(cls, split) for cls in split_table.index for split in split_table.columns if split_table.loc[cls, split] == 0]
    raise RuntimeError(f'At least one class has no images in a split: {zero_cells}. Review duplicate groups and source counts before training.')


## 6 — Create ImageFolder dataset

In [ ]:
# Rebuild split folders from scratch so reruns cannot leave stale images behind.
import shutil
for s in ['train','val','test']:
    split_dir=CLEAN/s
    if split_dir.exists(): shutil.rmtree(split_dir)
    for c in TARGET: (split_dir/c).mkdir(parents=True,exist_ok=True)
for _,r in tqdm(final.iterrows(),total=len(final),desc='Writing final images'):
    out=CLEAN/r['split']/r['class']/f"{r['class']}__{r.sha256[:16]}.jpg"
    with Image.open(r.original_path) as im: im.convert('RGB').save(out,quality=95)

dist=pd.DataFrame([{'split':s,'class':c,'count':len(list((CLEAN/s/c).glob('*.jpg')))} for s in ['train','val','test'] for c in TARGET])
dist.to_csv(META/'class_distribution.csv',index=False)
report={'raw_candidates':len(df),'valid_after_conflict_quarantine':len(valid),'invalid':len(bad),'conflicting_label_hashes_quarantined':int(len(conflict)),'conflicting_label_records_quarantined':int(len(conflict_rows)),'same_label_exact_duplicates_removed':len(valid)-len(clean),'final_images':len(final),'near_duplicate_candidates_within_class':len(near),'classes':TARGET}
(META/'dataset_report.json').write_text(json.dumps(report,indent=2))
assert final.sha256.nunique()==len(final)
assert (dist['count']>0).all()
assert int(dist['count'].sum())==len(final), 'On-disk image count differs from manifest.'
print(json.dumps(report,indent=2))


## 7 — MobileNetV3 compatibility check

In [ ]:
import torch
from torchvision import datasets,transforms

IM_SIZE=224
IMAGENET_MEAN=(0.485,0.456,0.406)
IMAGENET_STD=(0.229,0.224,0.225)
eval_tf=transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IM_SIZE), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])
train_tf=transforms.Compose([
    transforms.RandomResizedCrop(IM_SIZE,scale=(0.75,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.10),
    transforms.ToTensor(),transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])
train_ds=datasets.ImageFolder(CLEAN/'train',transform=train_tf)
val_ds=datasets.ImageFolder(CLEAN/'val',transform=eval_tf)
test_ds=datasets.ImageFolder(CLEAN/'test',transform=eval_tf)
assert train_ds.classes==val_ds.classes==test_ds.classes, 'Class order differs across splits.'
assert len(train_ds)>0 and len(val_ds)>0 and len(test_ds)>0
class_names=train_ds.classes
(META/'class_names.json').write_text(json.dumps(class_names,indent=2))
print('Device will be selected during training.')
print('Classes / output order:',class_names)
print('Train/Val/Test:',len(train_ds),len(val_ds),len(test_ds))
print('class_to_idx:',train_ds.class_to_idx)

## 8 — Create final ZIP

The ZIP contains `train/`, `val/`, `test/`, and `metadata/`.

In [ ]:
zip_path=Path('/content/Agrixore_Dataset_Final.zip')
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in CLEAN.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(CLEAN.parent))
print('Created:',zip_path)
print('Size GB:',round(zip_path.stat().st_size/1e9,2))


## 9 — Train MobileNetV3-Small

This section fine-tunes ImageNet-pretrained MobileNetV3-Small. It uses the validation set for checkpoint selection and early stopping; the test set is kept untouched until final evaluation. Training downloads pretrained weights, so internet access is required the first time. For reproducible results, use a GPU runtime when possible.

**Important:** only use the final test metrics after all model and threshold decisions are complete. Dataset performance is not a guarantee of field-level diagnostic accuracy.


In [ ]:
import copy, random, time, platform, numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=torch.cuda.is_available()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=32 if DEVICE.type=='cuda' else 16
NUM_WORKERS=2 if platform.system()!='Windows' else 0
EPOCHS=20
PATIENCE=5
LR_HEAD=1e-3
LR_BACKBONE=1e-4

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))

weights=MobileNet_V3_Small_Weights.DEFAULT
model=mobilenet_v3_small(weights=weights)
in_features=model.classifier[-1].in_features
model.classifier[-1]=nn.Linear(in_features,len(class_names))
model=model.to(DEVICE)

# Inverse-frequency weights reduce the influence of class imbalance.
counts=np.bincount([y for _,y in train_ds.samples],minlength=len(class_names)).astype(np.float32)
class_weights=(counts.sum()/(len(counts)*np.maximum(counts,1)))
criterion=nn.CrossEntropyLoss(weight=torch.tensor(class_weights,dtype=torch.float32,device=DEVICE))
optimizer=torch.optim.AdamW([
    {'params':model.features.parameters(),'lr':LR_BACKBONE},
    {'params':model.classifier.parameters(),'lr':LR_HEAD}],weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=2)

OUT=Path('/content/Agrixore_Model_Artifacts'); OUT.mkdir(parents=True,exist_ok=True)
best_path=OUT/'best_model_state.pt'

def run_epoch(loader,training):
    model.train(training)
    total_loss=0.0; y_true=[]; y_pred=[]
    context=torch.enable_grad() if training else torch.no_grad()
    with context:
        for xb,yb in tqdm(loader,leave=False):
            xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            logits=model(xb); loss=criterion(logits,yb)
            if training:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.0); optimizer.step()
            total_loss += loss.item()*yb.size(0)
            y_true.extend(yb.detach().cpu().tolist()); y_pred.extend(logits.argmax(1).detach().cpu().tolist())
    p,r,f1,_=precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
    return {'loss':total_loss/len(loader.dataset),'accuracy':accuracy_score(y_true,y_pred),'macro_precision':p,'macro_recall':r,'macro_f1':f1}

history=[]; best_f1=-1.0; best_epoch=0; stale=0
start=time.time()
for epoch in range(1,EPOCHS+1):
    train_metrics=run_epoch(train_loader,True)
    val_metrics=run_epoch(val_loader,False)
    scheduler.step(val_metrics['macro_f1'])
    row={'epoch':epoch,**{f'train_{k}':v for k,v in train_metrics.items()},**{f'val_{k}':v for k,v in val_metrics.items()}}
    history.append(row)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_metrics['loss']:.4f} acc {train_metrics['accuracy']:.4f} | val loss {val_metrics['loss']:.4f} acc {val_metrics['accuracy']:.4f} macro-F1 {val_metrics['macro_f1']:.4f}")
    if val_metrics['macro_f1'] > best_f1:
        best_f1=val_metrics['macro_f1']; best_epoch=epoch; stale=0
        torch.save({'model_state_dict':copy.deepcopy(model.state_dict()),'class_names':class_names,'image_size':IM_SIZE,'imagenet_mean':IMAGENET_MEAN,'imagenet_std':IMAGENET_STD,'best_epoch':best_epoch,'best_val_macro_f1':best_f1},best_path)
    else:
        stale+=1
        if stale>=PATIENCE:
            print(f'Early stopping at epoch {epoch}; best epoch was {best_epoch}.'); break

hist_df=pd.DataFrame(history); hist_df.to_csv(OUT/'training_history.csv',index=False)
checkpoint=torch.load(best_path,map_location=DEVICE,weights_only=False)
model.load_state_dict(checkpoint['model_state_dict']); model.eval()
print(f'Best checkpoint: epoch={best_epoch}, validation macro-F1={best_f1:.4f}, elapsed={time.time()-start:.1f}s')


## 10 — Learning curves and held-out test evaluation

The following cell loads the best validation checkpoint and evaluates it once on the held-out test set. Review per-class recall as well as overall accuracy: missed diseases can be more consequential than a false alarm.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(hist_df.epoch,hist_df.train_loss,label='Train loss'); axes[0].plot(hist_df.epoch,hist_df.val_loss,label='Validation loss'); axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(hist_df.epoch,hist_df.train_accuracy,label='Train accuracy'); axes[1].plot(hist_df.epoch,hist_df.val_accuracy,label='Validation accuracy'); axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

all_y=[]; all_pred=[]; all_prob=[]
with torch.no_grad():
    for xb,yb in tqdm(test_loader,desc='Held-out test'):
        logits=model(xb.to(DEVICE)); probs=torch.softmax(logits,dim=1).cpu().numpy()
        all_prob.extend(probs); all_pred.extend(probs.argmax(axis=1).tolist()); all_y.extend(yb.tolist())
all_prob=np.asarray(all_prob); all_y=np.asarray(all_y); all_pred=np.asarray(all_pred)
report=classification_report(all_y,all_pred,labels=list(range(len(class_names))),target_names=class_names,output_dict=True,zero_division=0)
print(classification_report(all_y,all_pred,labels=list(range(len(class_names))),target_names=class_names,zero_division=0,digits=4))
cm=confusion_matrix(all_y,all_pred,labels=list(range(len(class_names))))
fig,ax=plt.subplots(figsize=(10,8)); im=ax.imshow(cm,cmap='Blues'); ax.set_xticks(range(len(class_names)),class_names,rotation=75,ha='right'); ax.set_yticks(range(len(class_names)),class_names); ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix — held-out test'); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

metrics={'test_accuracy':float(accuracy_score(all_y,all_pred)),'test_macro_precision':float(precision_recall_fscore_support(all_y,all_pred,average='macro',zero_division=0)[0]),'test_macro_recall':float(precision_recall_fscore_support(all_y,all_pred,average='macro',zero_division=0)[1]),'test_macro_f1':float(precision_recall_fscore_support(all_y,all_pred,average='macro',zero_division=0)[2]),'best_epoch':int(best_epoch),'best_val_macro_f1':float(best_f1),'per_class':report}
(OUT/'evaluation_metrics.json').write_text(json.dumps(metrics,indent=2))
pd.DataFrame(cm,index=class_names,columns=class_names).to_csv(OUT/'confusion_matrix.csv')
print(json.dumps({k:v for k,v in metrics.items() if k!='per_class'},indent=2))


## 11 — Export deployment artifacts (PyTorch + ONNX)

Exports a TorchScript model and an ONNX model with a single input tensor shaped `[batch, 3, 224, 224]`, already RGB-converted, resized/cropped, converted to float, and normalized with ImageNet mean/std. For an in-browser PWA, ONNX Runtime Web can load `model.onnx`; the browser preprocessing must exactly match the configuration file. The ONNX model returns logits, so apply softmax and map indices using `class_names.json`.


In [ ]:
# Reload the best checkpoint one last time before export.
checkpoint=torch.load(best_path,map_location=DEVICE,weights_only=False)
model.load_state_dict(checkpoint['model_state_dict']); model.eval()

# TorchScript artifact (useful for Python/mobile pipelines).
torchscript_path=OUT/'model_torchscript.pt'
example=torch.zeros(1,3,IM_SIZE,IM_SIZE,device=DEVICE)
traced=torch.jit.trace(model,example)
traced.save(str(torchscript_path))

# ONNX artifact for browser inference with ONNX Runtime Web or server inference.
onnx_path=OUT/'model.onnx'
torch.onnx.export(model,example,onnx_path,export_params=True,opset_version=17,
                  do_constant_folding=True,input_names=['images'],output_names=['logits'],
                  dynamic_axes={'images':{0:'batch_size'},'logits':{0:'batch_size'}}, dynamo=False)

# Validate ONNX numerical parity when ONNX Runtime is available.
try:
    import onnx, onnxruntime as ort
    onnx_model=onnx.load(str(onnx_path)); onnx.checker.check_model(onnx_model)
    sess=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
    with torch.no_grad(): pt_logits=model(example).cpu().numpy()
    ort_logits=sess.run(['logits'],{'images':example.cpu().numpy()})[0]
    max_abs_diff=float(np.max(np.abs(pt_logits-ort_logits)))
    assert np.allclose(pt_logits,ort_logits,rtol=1e-3,atol=1e-4), f'ONNX parity check failed: max abs diff={max_abs_diff}'
    print('ONNX validation passed. Maximum absolute logit difference:',max_abs_diff)
except ImportError:
    max_abs_diff=None
    print('onnxruntime unavailable; skipped runtime parity check. Install onnxruntime and rerun export cell to validate.')

config={
 'model_name':'Agrixore MobileNetV3-Small', 'model_file':'model.onnx',
 'input_name':'images','output_name':'logits','input_layout':'NCHW',
 'input_dtype':'float32','input_color':'RGB','input_size':[IM_SIZE,IM_SIZE],
 'resize_shorter_side':256,'center_crop':[IM_SIZE,IM_SIZE],
 'normalization':{'mean':list(IMAGENET_MEAN),'std':list(IMAGENET_STD)},
 'class_names':class_names,'output_semantics':'logits; apply softmax for probabilities',
 'best_epoch':int(best_epoch),'best_validation_macro_f1':float(best_f1),
 'test_metrics':{k:v for k,v in metrics.items() if k!='per_class'},
 'onnx_max_abs_logit_difference':max_abs_diff,
 'limitations':'Predictions are limited to the listed classes and training-data domain. Confidence is not a calibrated probability of real-world correctness; request expert review for uncertain or severe cases.'
}
(OUT/'model_config.json').write_text(json.dumps(config,indent=2))
(OUT/'class_names.json').write_text(json.dumps(class_names,indent=2))

# Inference helper to smoke-test a PIL image using exactly the exported preprocessing.
def predict_pil(image):
    if not isinstance(image,Image.Image): image=Image.open(image)
    x=eval_tf(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): probs=torch.softmax(model(x),dim=1)[0].cpu().numpy()
    order=np.argsort(probs)[::-1]
    return [{'class':class_names[int(i)],'confidence':float(probs[i])} for i in order[:3]]

print('Exported artifacts:')
for f in sorted(OUT.iterdir()):
    if f.is_file(): print(f.name, f'{f.stat().st_size/1024/1024:.2f} MB')


## 12 — Smoke test, package, and download

Run this with a sample leaf image you have permission to use. Inspect the top-3 predictions and verify the PWA uses the exact same class order and preprocessing. Do not interpret the model as a plant-health authority; the app should communicate uncertainty and provide safe next steps.


In [ ]:
# Optional: set SAMPLE_IMAGE to a real leaf photo in the runtime to test end-to-end inference.
SAMPLE_IMAGE=None  # e.g. '/content/sample_leaf.jpg'
if SAMPLE_IMAGE:
    print(predict_pil(SAMPLE_IMAGE))
else:
    print('Set SAMPLE_IMAGE to a real leaf image to run the prediction smoke test.')

# Bundle model artifacts and reports. Dataset ZIP is created earlier in the notebook.
import zipfile
artifact_zip=Path('/content/Agrixore_Model_Artifacts.zip')
if artifact_zip.exists(): artifact_zip.unlink()
with zipfile.ZipFile(artifact_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file(): z.write(f,f.relative_to(OUT.parent))
print('Model artifact bundle:',artifact_zip,'size MB:',round(artifact_zip.stat().st_size/1e6,2))
print('Dataset bundle, if built:',Path('/content/Agrixore_Dataset_Final.zip').exists())


## Deployment checklist and limitations

- Upload `model.onnx`, `class_names.json`, and `model_config.json` to the PWA's static assets or model storage. Keep the model and class map versioned together.
- In-browser inference can use ONNX Runtime Web. The client must preprocess an uploaded image as RGB, resize the shorter side to 256, center-crop to 224×224, scale pixels to `[0,1]`, then normalize channel-wise with the exact mean/std in `model_config.json`.
- Apply softmax to logits, sort scores, and display the matching class name from `class_names.json`. A top score is not automatically a trustworthy probability; select and validate an abstention threshold using validation data before production.
- Add an explicit “unknown / unsupported image” flow. This classifier is a closed-set classifier and can force a disease label for weeds, non-leaf images, other crops, poor photos, or diseases outside the 13 classes.
- Test on independently collected local field images (ideally from Pakistan, different devices, lighting, growth stages, and cultivars) before making real-world claims. The public test split is not an external field validation set.
- Check source terms before distributing datasets or model weights. In particular, the listed cotton dataset has an unknown license. Dataset redistribution and model deployment are separate permission questions.
- This notebook was not executed end-to-end in this environment because it requires downloading external datasets and pretrained weights. Run all cells in a fresh Colab GPU runtime; review the printed source class counts and stop if any mapping/count is unexpected.


### PWA inference reference (JavaScript + ONNX Runtime Web)

Install `onnxruntime-web` in your frontend and serve `model.onnx` with your app assets or model storage. The example below demonstrates preprocessing for one uploaded `File`; for production, handle very large images, EXIF orientation, worker execution, model versioning, and low-confidence predictions. The model output is logits; the softmax and class order must match the notebook artifacts.

```js
import * as ort from 'onnxruntime-web';

const CLASS_NAMES = [
  'corn_blight', 'corn_common_rust', 'corn_gray_leaf_spot', 'corn_healthy',
  'cotton_bacterial_blight', 'cotton_curl_virus', 'cotton_fusarium_wilt', 'cotton_healthy',
  'wheat_brown_rust', 'wheat_healthy', 'wheat_mildew', 'wheat_septoria', 'wheat_yellow_rust'
]; // Replace this constant with the exact class_names.json array from your trained run.
const MEAN = [0.485, 0.456, 0.406];
const STD = [0.229, 0.224, 0.225];
const session = await ort.InferenceSession.create("/models/model.onnx", { executionProviders: ["wasm"] });

export async function predictLeaf(file) {
  const bitmap = await createImageBitmap(file);
  const scale = 256 / Math.min(bitmap.width, bitmap.height);
  const rw = Math.round(bitmap.width * scale), rh = Math.round(bitmap.height * scale);
  const resized = document.createElement("canvas"); resized.width = rw; resized.height = rh;
  const rctx = resized.getContext("2d", { willReadFrequently: true });
  rctx.drawImage(bitmap, 0, 0, rw, rh);
  const crop = document.createElement("canvas"); crop.width = 224; crop.height = 224;
  const cctx = crop.getContext("2d", { willReadFrequently: true });
  cctx.drawImage(resized, Math.floor((rw - 224) / 2), Math.floor((rh - 224) / 2), 224, 224, 0, 0, 224, 224);
  const pixels = cctx.getImageData(0, 0, 224, 224).data;
  const input = new Float32Array(1 * 3 * 224 * 224);
  for (let i = 0; i < 224 * 224; i++) {
    for (let c = 0; c < 3; c++) {
      const value = pixels[i * 4 + c] / 255;
      input[c * 224 * 224 + i] = (value - MEAN[c]) / STD[c];
    }
  }
  const tensor = new ort.Tensor("float32", input, [1, 3, 224, 224]);
  const result = await session.run({ images: tensor });
  const logits = result.logits.data;
  const maxLogit = Math.max(...logits);
  const exps = Array.from(logits, x => Math.exp(x - maxLogit));
  const sum = exps.reduce((a, b) => a + b, 0);
  return exps.map((x, i) => ({ disease: CLASS_NAMES[i], score: x / sum }))
    .sort((a, b) => b.score - a.score).slice(0, 3);
}
```

**Important:** the class array above is only an illustrative placeholder order. Always fetch/use the generated `class_names.json` from the same training run as `model.onnx`; never rely on a manually copied class list. Validate the JS output against the notebook `predict_pil()` function on the same sample images before release.
